BERTの事前学習タスク（推論だけ）を体験するコード

1. Masked language model  
2. Next sentence prediction  
3. Tokenizer / 前処理＝入力をモデル用の数値に変換

In [2]:
from transformers import BertTokenizer, BertForMaskedLM, BertForNextSentencePrediction
import torch


c:\Users\Nutzer\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\__init__.py:16: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.2)
  from scipy.sparse import issparse
c:\Users\Nutzer\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [ ]:
# p361〜363 (PyTorch) のBERTコード：実行できる版
# 必要: pip install torch transformers


# ----------------------------
# p361: 事前学習済みの BERT モデルと Tokenizer をロードします。
# ----------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
mlm_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
nsp_model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased")

# ----------------------------
# p362: コード4-大問15-1：事前学習 (PyTorch)
# ----------------------------
text_a = "I enjoy walking with my cute dog."
text_b = "He's very playful."

# --- MLM (Masked Language Model) セクション ---

# ターゲットとなる単語をマスクします。
masked_index = 4
tokens = tokenizer.tokenize(text_a)
tokens[masked_index] = "[MASK]"

# 入力をエンコードし、テンソルに変換します。
# "return_tensors='pt'" は PyTorch で使用する為の設定
# ★動くための最小修正:
#   tokens は「すでに分割済み(tokenized)」なので is_split_into_words=True を付けます。
encoded_input_mlm = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")

with torch.no_grad():
    output = mlm_model(**encoded_input_mlm)
    predictions = output.logits

# 予測されたトークンIDを取得し、トークンに変換します。
# ★動くための最小修正:
#   [CLS]等で index がズレるので、MASKの位置を実際に探します。
mask_pos = (encoded_input_mlm["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1].item()
predicted_token_id = torch.argmax(predictions[0, mask_pos]).item()
predicted_token = tokenizer.convert_ids_to_tokens([predicted_token_id])[0]

print("=== MLM ===")
print("Original tokens:", tokenizer.tokenize(text_a))
print("Masked tokens  :", tokens)
print("Predicted token:", predicted_token)

# --- NSP (Next Sentence Prediction) セクション ---

# 二つのテキストをエンコードし、テンソルに変換します。
# "return_tensors='pt'" は PyTorch で使用する為の設定
encoded_input_nsp = tokenizer(
    text_a,
    text_b,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512,
)

with torch.no_grad():
    output = nsp_model(**encoded_input_nsp)
    logits = output.logits

# 予測されたクラスを取得し、二つのセンテンスが連続しているかどうかを判断します。
is_next = torch.argmax(logits, dim=1).item() == 0

print("\n=== NSP ===")
print("is_next:", is_next)

# ----------------------------
# p363: コード4-大問15-2：tokenizer (PyTorch)
# ----------------------------
print("\n=== Tokenizer (code 15-2) ===")

# トークン化するテキスト
text = "Hello, BERT! How are you?"

# トークンに分割
token = tokenizer.tokenize(text)
print(f"Token: {token}")

# トークンを ID に変換
# ★動くための最小修正:
#   写真だと tokenizer.encode(token) になっているが、token が list なのでエラーになりやすい。
#   出力例(先頭101など)に合わせて、encodeは text を渡します。
token_ids = tokenizer.encode(text)
print(f"Token_IDs: {token_ids}")

# ID をトークンに変換
tokens_back = tokenizer.convert_ids_to_tokens(token_ids)
print(f"Back to tokens: {tokens_back}")

# ----------------------------
# p363: コード4-大問15-3：前処理 (PyTorch)
# ----------------------------
print("\n=== Preprocess (code 15-3) ===")

encoding = tokenizer(
    text,
    max_length=35, #トークン化された滝ストの最大長を35トークンに制限する
    padding="max_length", #トークンの数がmax_lengthに達していない場合に、残りの部分をパディングトークン([PAD])で埋める。これにより、すべてのエンコードされたテキストが同じ長さになり、バッチ処理が容易になる。
    truncation=True, #トークンの数がmax_lengthを超えている場合に、それをmax_lengthの長さに切り詰める動作を指定する。これにより、モデルが受け入れることができる最大のシーケンス超を超えないようにすることができる。
    return_tensors="pt", #エンコードされたテキストをpytorchのテンソル形式で返す動作を指定する。デフォルトは、エンコードされたテキストはPythonの辞書形式で返す。
)
print(encoding)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\Nutzer\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nutzer\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


=== MLM ===
Original tokens: ['i', 'enjoy', 'walking', 'with', 'my', 'cute', 'dog', '.']
Masked tokens  : ['i', 'enjoy', 'walking', 'with', '[MASK]', 'cute', 'dog', '.']
Predicted token: my

=== NSP ===
is_next: True

=== Tokenizer (code 15-2) ===
Token: ['hello', ',', 'bert', '!', 'how', 'are', 'you', '?']
Token_IDs: [101, 7592, 1010, 14324, 999, 2129, 2024, 2017, 1029, 102]
Back to tokens: ['[CLS]', 'hello', ',', 'bert', '!', 'how', 'are', 'you', '?', '[SEP]']

=== Preprocess (code 15-3) ===
{'input_ids': tensor([[  101,  7592,  1010, 14324,   999,  2129,  2024,  2017,  1029,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,

In [4]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

text = "Hello, BERT! How are you?"

token = tokenizer.tokenize(text)
print("Token:", token)

token_ids = tokenizer.encode(text)  # 先頭に[CLS]=101、末尾に[SEP]=102が付く
print("Token_IDs:", token_ids)


Token: ['hello', ',', 'bert', '!', 'how', 'are', 'you', '?']
Token_IDs: [101, 7592, 1010, 14324, 999, 2129, 2024, 2017, 1029, 102]
